In [1]:
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: Setup & Data Creation
# ==========================================

def create_messy_dataset(rows=100):
    """
    Creates a dataset with intentional issues: duplicates, missing values,
    and incorrect data types for demonstration purposes.
    """
    np.random.seed(42)

    data = {
        'EmployeeID': np.random.randint(1000, 9999, rows),
        'Name': [f'Employee_{i}' for i in range(rows)],
        'Age': np.random.randint(22, 60, rows).astype(float),
        'Salary': np.random.randint(40000, 120000, rows).astype(str),  # Incorrect type (String)
        'Department': np.random.choice(['HR', 'Engineering', 'Sales', 'Marketing'], rows)
    }

    df = pd.DataFrame(data)

    # 1.1: Intentionally corrupt 'Salary' to look like currency strings
    # Adding '$' and ',' to make it a string type that requires cleaning
    df['Salary'] = df['Salary'].apply(lambda x: f"${x[:2]},{x[2:]}")

    # 1.2: Intentionally insert Missing Values (NaN)
    # Set 10% of Age values to NaN
    df.loc[df.sample(frac=0.1).index, 'Age'] = np.nan
    # Set 10% of Department values to NaN
    df.loc[df.sample(frac=0.1).index, 'Department'] = np.nan

    # 1.3: Intentionally create Duplicate Rows
    # Duplicate the first 5 rows and append them
    duplicates = df.head(5)
    df = pd.concat([df, duplicates], ignore_index=True)

    print(f"Dataset Created. Shape: {df.shape}")
    return df

# Initialize the dataframe
df = create_messy_dataset()

print("\n--- Raw Data (First 5 Rows) ---")
print(df.head())
print("\n--- Data Info (Before Cleaning) ---")
print(df.info())

# ==========================================
# STEP 2: Data Cleaning Process
# ==========================================

# 2.1: Remove Duplicate Rows
initial_rows = len(df)
df_cleaned = df.drop_duplicates()
print(f"\n[Cleaning] Removed {initial_rows - len(df_cleaned)} duplicate rows.")

# 2.2: Convert Incorrect Data Types
# Removing '$' and ',' from Salary and converting to Float
# We use regex=True to replace non-numeric characters
try:
    df_cleaned = df_cleaned.copy() # Avoid SettingWithCopyWarning
    df_cleaned['Salary'] = df_cleaned['Salary'].str.replace(r'[$,]', '', regex=True).astype(float)
    print("[Cleaning] Successfully converted 'Salary' column to numeric.")
except Exception as e:
    print(f"[Error] Failed to convert Salary: {e}")

# 2.3: Handle Missing Values
# Strategy: Mean for Numerical (Age), Mode for Categorical (Department)

# Handling 'Age' (Numerical) -> Fill with Mean
mean_age = df_cleaned['Age'].mean()
df_cleaned['Age'] = df_cleaned['Age'].fillna(mean_age)
print(f"[Cleaning] Filled missing 'Age' values with Mean: {mean_age:.2f}")

# Handling 'Department' (Categorical) -> Fill with Mode
mode_dept = df_cleaned['Department'].mode()[0]
df_cleaned['Department'] = df_cleaned['Department'].fillna(mode_dept)
print(f"[Cleaning] Filled missing 'Department' values with Mode: {mode_dept}")

# ==========================================
# STEP 3: Verification & Saving
# ==========================================

print("\n--- Data Info (After Cleaning) ---")
print(df_cleaned.info())

print("\n--- Cleaned Data (First 5 Rows) ---")
print(df_cleaned.head())

# Save to CSV
output_filename = "cleaned_employee_data.csv"
df_cleaned.to_csv(output_filename, index=False)
print(f"\n[Success] Cleaned dataset saved to '{output_filename}'")

Dataset Created. Shape: (105, 5)

--- Raw Data (First 5 Rows) ---
   EmployeeID        Name   Age    Salary Department
0        8270  Employee_0  56.0   $48,680  Marketing
1        1860  Employee_1  22.0  $11,1295      Sales
2        6390  Employee_2  56.0   $51,111  Marketing
3        6191  Employee_3  58.0   $77,504        NaN
4        6734  Employee_4  35.0   $41,802         HR

--- Data Info (Before Cleaning) ---
<class 'pandas.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   EmployeeID  105 non-null    int32  
 1   Name        105 non-null    str    
 2   Age         95 non-null     float64
 3   Salary      105 non-null    str    
 4   Department  94 non-null     str    
dtypes: float64(1), int32(1), str(3)
memory usage: 3.8 KB
None

[Cleaning] Removed 5 duplicate rows.
[Cleaning] Successfully converted 'Salary' column to numeric.
[Cleaning] Filled missing 'Age' valu